In [ ]:
import os
import h5py
import numpy as np
from pathlib import Path
import pyFAI
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

PATTERNS_PATH = r""
LABELS_DIR = r""
OUTPUT_PATH = r""
TEST_SPLIT_RATIO = 0.2
RANDOM_STATE = 42

print("Finding matching patterns and labels...")
hashes = []
if os.path.exists(PATTERNS_PATH):
    with h5py.File(PATTERNS_PATH, 'r') as hf:
        for ds_name in hf.keys():
            # In case paths have leading slashes, strip them for the filename
            ds_str = str(ds_name).strip('/')
            poni_path = Path(LABELS_DIR) / f"{ds_str}.poni"
            if poni_path.exists():
                hashes.append(ds_name)
    print(f"Found {len(hashes)} matches.")
else:
    print(f"File not found: {PATTERNS_PATH}")

In [ ]:
def read_poni_label(poni_path):
    ai = pyFAI.load(str(poni_path))
    return np.array([
        ai.dist,
        ai.poni1,
        ai.poni2,
        ai.rot1,
        ai.rot2,
        ai.rot3
    ], dtype=np.float32)

if hashes:
    train_hashes, test_hashes = train_test_split(hashes, test_size=TEST_SPLIT_RATIO, random_state=RANDOM_STATE)

    def write_group(group, current_hashes, patterns_h5):
        if not current_hashes:
            return
     
        sample_image = patterns_h5[current_hashes[0]][...]
        image_shape = sample_image.shape
        num_samples = len(current_hashes)
        
        print(f"Writing {num_samples} samples to group '{group.name}'...")
        dset_images = group.create_dataset("images", (num_samples, *image_shape), dtype='f4', chunks=(1, *image_shape))
        dset_labels = group.create_dataset("labels", (num_samples, 6), dtype='f4', chunks=(1, 6))
        
        for i, h in enumerate(current_hashes):
            dset_images[i] = patterns_h5[h][...].astype(np.float32)
            h_str = str(h).strip('/')
            dset_labels[i] = read_poni_label(Path(LABELS_DIR) / f"{h_str}.poni")

    print(f"Writing structured dataset to {OUTPUT_PATH}...")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    with h5py.File(PATTERNS_PATH, 'r') as hf_in, h5py.File(OUTPUT_PATH, 'w') as hf_out:
        train_group = hf_out.create_group('train')
        test_group = hf_out.create_group('test')
        
        write_group(train_group, train_hashes, hf_in)
        write_group(test_group, test_hashes, hf_in)

        train_labels_array = train_group['labels'][:] 
        
        centers = np.mean(train_labels_array, axis=0)
        scale_factors = np.std(train_labels_array, axis=0)
        scale_factors[scale_factors == 0.0] = 1.0

        norm_group = hf_out.create_group("normalization")
        norm_group.create_dataset("centers", data=centers.astype(np.float32))
        norm_group.create_dataset("scale_factors", data=scale_factors.astype(np.float32))

    print("Formatting complete.")
else:
    print("No valid matches found to format.")